# ETF_V4：固定当前正交 ETF 池构建器

目标：先用“当前可投 ETF 池”生成一份固定、版本化、相对正交的 ETF 池，然后训练、回测、未来模拟都使用同一份池子。

这个实验的口径是：

> 如果今天确定一套可实盘 ETF 正交池，那么这套池子在历史上可交易时，模型表现如何。

因此它不是“还原历史全 ETF 宇宙”的研究，而是“当前实盘可投池”的历史回放。为了避免最明显的穿越问题，后续训练/回测仍然会按历史日期过滤：当时已上市、上市满 180 天、有价格、有成交额。

本版参数是 v1b：目标不是极简 60 只代表池，而是保留大约 120-180 只、适合 ML 横截面排序的固定正交池。

输出：

- `etf_orthogonal_pool_v1.csv`：最终保留池
- `etf_orthogonal_pool_dropped_v1.csv`：被剔除/同质替代池
- `etf_orthogonal_pool_group_summary_v1.csv`：分组统计
- `etf_orthogonal_pool_corr_pairs_v1.csv`：高相关 ETF 对

In [ ]:
# =========================
# 0. Config
# =========================
from jqdata import *
import os
import math
import datetime
import numpy as np
import pandas as pd

try:
    from tqdm import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

OUT_DIR = "etf_v4_fixed_orthogonal_pool_outputs"
POOL_CSV = os.path.join(OUT_DIR, "etf_orthogonal_pool_v1.csv")
DROPPED_CSV = os.path.join(OUT_DIR, "etf_orthogonal_pool_dropped_v1.csv")
GROUP_SUMMARY_CSV = os.path.join(OUT_DIR, "etf_orthogonal_pool_group_summary_v1.csv")
CORR_PAIRS_CSV = os.path.join(OUT_DIR, "etf_orthogonal_pool_corr_pairs_v1.csv")

AS_OF_DATE = "2026-06-26"
MIN_LISTING_DAYS = 180
MIN_AVG_MONEY_20 = 10000000.0
MIN_AVG_MONEY_60 = 5000000.0
CORR_LOOKBACK_DAYS = 250
CORR_THRESHOLD = 0.97
ETF_CHUNK_SIZE = 120
POOL_VERSION = "etf_orthogonal_pool_v1"

EXCLUDE_NAME_KEYWORDS = [
    "债", "国债", "地债", "政金债", "公司债", "城投", "可转债",
    "货币", "现金", "快线", "快钱", "同业存单", "REIT", "REITS",
    "AAA", "信用", "短融", "中票", "0-3", "1-3", "政策性金融债",
]

# group limit 是建池阶段的代表 ETF 数量，不是回测阶段的持仓行业限制。
GROUP_KEEP_LIMITS = {
    # This is the fixed investable universe size, not a portfolio holding cap.
    # Keep it broad enough for ML cross-sectional ranking.
    "broad_a": 25,
    "broad_gem_star": 16,
    "hk_broad": 16,
    "us_global": 12,
    "semiconductor": 8,
    "software_ai": 10,
    "innovative_drug": 8,
    "bank": 5,
    "broker": 5,
    "new_energy": 8,
    "gold": 4,
    "oil_gas": 4,
    "coal": 3,
    "nonferrous": 5,
    "consumer": 8,
    "military": 5,
    "real_estate": 5,
    "dividend_value": 8,
    "other_equity": 30,
}
DEFAULT_GROUP_KEEP_LIMIT = 5

GROUP_RULES = [
    ("dividend_value", ["红利", "低波", "价值", "央企"]),
    ("broad_gem_star", ["创业板", "科创", "双创", "创50", "科创50", "创业板50", "创业板200"]),
    ("broad_a", ["沪深300", "中证500", "中证800", "中证1000", "A500", "A50", "上证50", "MSCI", "深证100"]),
    ("hk_broad", ["恒生", "港股通", "港股", "H股", "中国互联", "恒科技"]),
    ("us_global", ["标普", "纳指", "纳斯达克", "日经", "德国", "法国", "东南亚", "海外", "美国"]),
    ("semiconductor", ["半导", "芯片", "集成电路"]),
    ("software_ai", ["软件", "人工智能", "AI", "云计算", "大数据", "计算机", "信创", "游戏", "传媒", "通信", "5G"]),
    ("innovative_drug", ["创新药", "新药", "医药", "医疗", "生物", "药"]),
    ("bank", ["银行"]),
    ("broker", ["证券", "券商"]),
    ("new_energy", ["新能源", "光伏", "电池", "锂电", "储能", "电力设备"]),
    ("gold", ["黄金"]),
    ("oil_gas", ["油气", "石油", "能源"]),
    ("coal", ["煤炭"]),
    ("nonferrous", ["有色", "稀有金属", "稀土", "钢铁"]),
    ("consumer", ["消费", "食品", "酒", "家电", "农业", "畜牧"]),
    ("military", ["军工", "国防", "航空航天", "通用航空"]),
    ("real_estate", ["地产", "房地产", "基建", "建筑"]),
]
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

print("out dir:", OUT_DIR)
print("as_of_date:", AS_OF_DATE)
print("corr threshold:", CORR_THRESHOLD)

In [ ]:
# =========================
# 1. Helpers
# =========================
def chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def should_exclude_name(name):
    text = str(name)
    upper = text.upper()
    for kw in EXCLUDE_NAME_KEYWORDS:
        if kw and (kw in text or kw.upper() in upper):
            return True
    return False


def classify_etf_group(name):
    text = str(name)
    upper = text.upper()
    for group_name, keywords in GROUP_RULES:
        for kw in keywords:
            if kw and (kw in text or kw.upper() in upper):
                return group_name
    return "other_equity"


def get_current_trade_date(as_of_date):
    try:
        days = pd.to_datetime(get_trade_days(end_date=as_of_date, count=1))
    except NameError:
        raise RuntimeError("本 notebook 需要在 JoinQuant 研究环境运行。")
    if len(days) == 0:
        raise RuntimeError("no trade day found before " + str(as_of_date))
    return pd.Timestamp(days[-1]).normalize()


def get_current_etf_metadata(as_of_date):
    try:
        sec_df = get_all_securities(["etf"], date=as_of_date)
    except NameError:
        raise RuntimeError("get_all_securities 不可用：请在 JoinQuant 研究环境运行。")
    if sec_df is None or sec_df.empty:
        raise RuntimeError("empty ETF securities table")
    rows = []
    asof = pd.Timestamp(as_of_date).date()
    for code, row in sec_df.iterrows():
        name = str(row.get("display_name", ""))
        start_date = row.get("start_date", None)
        try:
            if pd.isnull(start_date):
                start_date = get_security_info(code).start_date
            start_date = pd.Timestamp(start_date).date()
        except Exception:
            start_date = None
        listed_days = (asof - start_date).days if start_date is not None else np.nan
        exclude = should_exclude_name(name)
        rows.append({
            "code": code,
            "name": name,
            "start_date": start_date,
            "listed_days": listed_days,
            "name_excluded": bool(exclude),
            "group": classify_etf_group(name),
        })
    return pd.DataFrame(rows)


def fetch_liquidity_and_returns(codes, as_of_date):
    rows = []
    close_parts = []
    fields = ["close", "money"]
    count = max(CORR_LOOKBACK_DAYS + 1, 61)
    for code_chunk in tqdm(list(chunks(codes, ETF_CHUNK_SIZE)), desc="fetch etf price"):
        try:
            px = get_price(
                code_chunk,
                end_date=as_of_date,
                frequency="daily",
                fields=fields,
                count=count,
                panel=False,
                fq="pre",
                skip_paused=False,
            )
        except Exception as err:
            print("price chunk failed", err)
            continue
        if px is None or px.empty:
            continue
        px["time"] = pd.to_datetime(px["time"]).dt.normalize()
        for code, one in px.groupby("code"):
            one = one.sort_values("time").copy()
            close = pd.Series(one["close"].astype(float).values)
            money = pd.Series(one["money"].astype(float).values)
            rows.append({
                "code": code,
                "price_days": len(one),
                "avg_money_20": money.tail(20).mean(),
                "avg_money_60": money.tail(60).mean(),
                "ret_20": close.iloc[-1] / close.iloc[-21] - 1.0 if len(close) > 20 and close.iloc[-21] > 0 else np.nan,
                "ret_60": close.iloc[-1] / close.iloc[-61] - 1.0 if len(close) > 60 and close.iloc[-61] > 0 else np.nan,
                "vol_60": close.pct_change().tail(60).std() if len(close) > 60 else np.nan,
            })
            close_parts.append(one[["time", "code", "close"]].copy())
    liq_df = pd.DataFrame(rows)
    if close_parts:
        close_df = pd.concat(close_parts, ignore_index=True)
        close_mat = close_df.pivot_table(index="time", columns="code", values="close").sort_index()
        ret_mat = close_mat.pct_change().tail(CORR_LOOKBACK_DAYS)
    else:
        ret_mat = pd.DataFrame()
    return liq_df, ret_mat


def quality_score(row):
    money20 = row.get("avg_money_20", 0.0)
    money60 = row.get("avg_money_60", 0.0)
    listed = row.get("listed_days", 0.0)
    price_days = row.get("price_days", 0.0)
    score = 0.0
    score += math.log(max(float(money20), 1.0)) * 2.0
    score += math.log(max(float(money60), 1.0)) * 1.0
    score += min(float(listed) if not pd.isnull(listed) else 0.0, 2500.0) / 2500.0
    score += min(float(price_days) if not pd.isnull(price_days) else 0.0, float(CORR_LOOKBACK_DAYS)) / float(CORR_LOOKBACK_DAYS)
    return score


def find_corr_clusters(group_df, ret_mat):
    codes = [c for c in group_df["code"].tolist() if c in ret_mat.columns]
    if len(codes) == 0:
        out = group_df.copy()
        out["corr_cluster"] = ["no_price_%s" % i for i in range(len(out))]
        return out, []
    corr = ret_mat[codes].corr(min_periods=80)
    parent = dict((c, c) for c in group_df["code"].tolist())

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    pair_rows = []
    for i, c1 in enumerate(codes):
        for c2 in codes[i + 1:]:
            v = corr.loc[c1, c2]
            if pd.isnull(v):
                continue
            if float(v) >= CORR_THRESHOLD:
                union(c1, c2)
                pair_rows.append({"code1": c1, "code2": c2, "corr": float(v)})
    out = group_df.copy()
    out["corr_cluster"] = out["code"].map(lambda x: find(x) if x in parent else x)
    return out, pair_rows

In [ ]:
# =========================
# 2. Build candidate universe
# =========================
as_of_trade_date = get_current_trade_date(AS_OF_DATE)
print("as-of trade date:", as_of_trade_date)

meta_df = get_current_etf_metadata(as_of_trade_date)
print("raw etfs:", meta_df.shape)

base_df = meta_df[(~meta_df["name_excluded"]) & (meta_df["listed_days"] >= MIN_LISTING_DAYS)].copy()
print("after name/listing filters:", base_df.shape)

display(base_df.groupby("group")["code"].count().sort_values(ascending=False).to_frame("raw_count"))
display(base_df.head())

In [ ]:
# =========================
# 3. Liquidity, returns, and current tradable filter
# =========================
liq_df, ret_mat = fetch_liquidity_and_returns(base_df["code"].tolist(), as_of_trade_date)
base_df = base_df.merge(liq_df, on="code", how="left")
base_df["liquidity_ok"] = (base_df["avg_money_20"] >= MIN_AVG_MONEY_20) & (base_df["avg_money_60"] >= MIN_AVG_MONEY_60)
base_df["price_ok"] = base_df["price_days"] >= 80
candidate_df = base_df[base_df["liquidity_ok"] & base_df["price_ok"]].copy()
candidate_df["quality_score"] = candidate_df.apply(quality_score, axis=1)

print("candidate etfs:", candidate_df.shape)
print("return matrix:", ret_mat.shape)
display(candidate_df.groupby("group")["code"].count().sort_values(ascending=False).to_frame("candidate_count"))
display(candidate_df.sort_values("quality_score", ascending=False).head(20))

In [ ]:
# =========================
# 4. Correlation clustering and representative selection
# =========================
kept_parts = []
dropped_parts = []
corr_pair_parts = []

for group_name, gdf in tqdm(candidate_df.groupby("group"), desc="orthogonal groups"):
    gdf = gdf.sort_values("quality_score", ascending=False).copy()
    clustered, pair_rows = find_corr_clusters(gdf, ret_mat)
    if pair_rows:
        for r in pair_rows:
            r["group"] = group_name
        corr_pair_parts.append(pd.DataFrame(pair_rows))

    group_limit = GROUP_KEEP_LIMITS.get(group_name, DEFAULT_GROUP_KEEP_LIMIT)
    kept_codes = []
    dropped_rows = []
    keep_rows = []
    for cluster_id, cdf in clustered.groupby("corr_cluster"):
        cdf = cdf.sort_values("quality_score", ascending=False).copy()
        winner = cdf.iloc[0].copy()
        winner["keep_reason"] = "best_liquidity_in_corr_cluster"
        keep_rows.append(winner)
        if len(cdf) > 1:
            losers = cdf.iloc[1:].copy()
            losers["drop_reason"] = "high_corr_duplicate_of_%s" % winner["code"]
            dropped_rows.append(losers)
    group_keep = pd.DataFrame(keep_rows).sort_values("quality_score", ascending=False).copy()
    group_keep["keep_rank_in_group"] = np.arange(1, len(group_keep) + 1)
    group_keep["keep_flag"] = group_keep["keep_rank_in_group"] <= group_limit
    kept_parts.append(group_keep[group_keep["keep_flag"]].copy())

    over_limit = group_keep[~group_keep["keep_flag"]].copy()
    if not over_limit.empty:
        over_limit["drop_reason"] = "group_keep_limit_%s" % group_limit
        dropped_rows.append(over_limit)
    if dropped_rows:
        dropped_parts.append(pd.concat(dropped_rows, ignore_index=True))

pool_df = pd.concat(kept_parts, ignore_index=True) if kept_parts else pd.DataFrame()
dropped_df = pd.concat(dropped_parts, ignore_index=True) if dropped_parts else pd.DataFrame()
corr_pairs_df = pd.concat(corr_pair_parts, ignore_index=True) if corr_pair_parts else pd.DataFrame(columns=["group", "code1", "code2", "corr"])

pool_df["pool_version"] = POOL_VERSION
pool_df["as_of_date"] = as_of_trade_date
pool_df = pool_df.sort_values(["group", "keep_rank_in_group", "quality_score"], ascending=[True, True, False]).copy()

cols = [
    "pool_version", "as_of_date", "code", "name", "group", "corr_cluster", "keep_rank_in_group",
    "start_date", "listed_days", "avg_money_20", "avg_money_60", "price_days",
    "ret_20", "ret_60", "vol_60", "quality_score", "keep_reason",
]
pool_df = pool_df[[c for c in cols if c in pool_df.columns]]

print("final pool size:", pool_df.shape)
display(pool_df)

In [ ]:
# =========================
# 5. Diagnostics and save outputs
# =========================
group_summary_df = pool_df.groupby("group").agg({
    "code": "count",
    "avg_money_20": "median",
    "avg_money_60": "median",
}).reset_index()
group_summary_df = group_summary_df.rename(columns={
    "code": "keep_count",
    "avg_money_20": "avg_money_20",
    "avg_money_60": "avg_money_60",
})
group_summary_df = group_summary_df.sort_values("keep_count", ascending=False)

# Attach readable names to high-correlation pairs.
if not corr_pairs_df.empty:
    name_map = dict(zip(candidate_df["code"], candidate_df["name"]))
    corr_pairs_df["name1"] = corr_pairs_df["code1"].map(name_map)
    corr_pairs_df["name2"] = corr_pairs_df["code2"].map(name_map)
    corr_pairs_df = corr_pairs_df.sort_values("corr", ascending=False)

pool_df.to_csv(POOL_CSV, index=False)
dropped_df.to_csv(DROPPED_CSV, index=False)
group_summary_df.to_csv(GROUP_SUMMARY_CSV, index=False)
corr_pairs_df.to_csv(CORR_PAIRS_CSV, index=False)

print("saved:")
for p in [POOL_CSV, DROPPED_CSV, GROUP_SUMMARY_CSV, CORR_PAIRS_CSV]:
    print(" -", p)
print("group summary:")
display(group_summary_df)
print("top corr pairs:")
display(corr_pairs_df.head(30))
print("dropped sample:")
display(dropped_df.head(30))

## Self Review

- 本 notebook 使用当前 ETF 池做固定正交池，目的是服务未来实盘可投 universe，不声称还原历史全市场 ETF 机会集。
- 输出池子后，训练和回测仍需按历史日期检查上市天数、行情和成交额，避免把未上市 ETF 强行带入历史。
- 正交池构建使用名称分组 + 过去收益相关性聚类 + 流动性代表选择；不是回测阶段的行业硬限制。
- `other_equity` 组需要人工 review，如果里面混入重要主题，后续应补充 `GROUP_RULES` 后重建池子。
- 池子是版本化文件；后续新增 ETF 或流动性变化明显时再升级为 `v2`，不要静默覆盖。